# Narrador no Google Colab: slides narrados com a SUA voz clonada (GPU grátis)

Este notebook roda o **Narrador** com o motor **Chatterbox Multilingual pt-BR** (MIT, gratuito)
na GPU gratuita do Colab. Você só precisa de:

1. um arquivo `.pptx` (com notas do orador) ou `.pdf` + roteiro `.md`;
2. uma gravação da sua voz de **8 a 15 segundos** (mp3/m4a/wav), limpa e sem música.

**Antes de rodar:** menu *Ambiente de execução → Alterar o tipo de ambiente → GPU (T4)*.
Execute as células em ordem (Shift+Enter).

In [ ]:
#@title 1. Instalar dependências (2-4 min)
!apt-get -qq install -y ffmpeg libreoffice-impress > /dev/null
!git clone -q https://github.com/mscpatricio/OBRAS-SAUDE.git /content/repo
%pip install -q /content/repo/narrador chatterbox-tts huggingface_hub
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NENHUMA (ative a GPU no menu Ambiente de execução)")

In [ ]:
#@title 2. Enviar a gravação da sua voz
from google.colab import files
import shutil, pathlib
up = files.upload()
nome = next(iter(up))
pathlib.Path("/content/voz").mkdir(exist_ok=True)
shutil.move(nome, f"/content/voz/{nome}")
transcricao = "" #@param {type:"string"}
!narrador voz preparar "/content/voz/{nome}" --saida /content/voz/minha_voz.wav {"--transcricao \"" + transcricao + "\"" if transcricao else ""}
from IPython.display import Audio; Audio("/content/voz/minha_voz.wav")

In [ ]:
#@title 3. Testar a voz clonada (ouça antes de narrar tudo)
texto_teste = "Olá! Esta é uma amostra da minha voz clonada. Vamos acompanhar as obras de saúde." #@param {type:"string"}
exaggeration = 0.45 #@param {type:"slider", min:0.2, max:0.9, step:0.05}
cfg_weight = 0.5 #@param {type:"slider", min:0.2, max:0.8, step:0.05}
!narrador voz testar --motor chatterbox-ptbr --voz /content/voz/minha_voz.wav -t "{texto_teste}" \
   -o exaggeration={exaggeration} -o cfg_weight={cfg_weight} -s /content/saida/teste_voz.wav
from IPython.display import Audio; Audio("/content/saida/teste_voz.wav")

In [ ]:
#@title 4. Enviar a apresentação (.pptx ou .pdf) e, se quiser, o roteiro .md
from google.colab import files
import shutil, pathlib
pathlib.Path("/content/entrada").mkdir(exist_ok=True)
up = files.upload()
for n in up:
    shutil.move(n, f"/content/entrada/{n}")
print("Arquivos:", [p.name for p in pathlib.Path("/content/entrada").iterdir()])

In [ ]:
#@title 5. Gerar o vídeo narrado
import pathlib, glob
apres = next(p for p in pathlib.Path("/content/entrada").iterdir() if p.suffix.lower() in {".pptx", ".pdf"})
roteiros = glob.glob("/content/entrada/*.md")
rot = f'--roteiro "{roteiros[0]}"' if roteiros else ""
legendas = True #@param {type:"boolean"}
movimento = False #@param {type:"boolean"}
transicao = 0.4 #@param {type:"number"}
flags = (" --legendas" if legendas else "") + (" --movimento" if movimento else "")
!narrador gerar "{apres}" {rot} --motor chatterbox-ptbr --voz /content/voz/minha_voz.wav \
   -o exaggeration={exaggeration} -o cfg_weight={cfg_weight} --transicao {transicao}{flags} --saida /content/saida
mp4 = glob.glob("/content/saida/*/*.mp4")[0]; print("Vídeo:", mp4)

In [ ]:
#@title 6. Baixar o MP4 (e as legendas .srt)
from google.colab import files
import glob
mp4 = glob.glob("/content/saida/*/*.mp4")[0]
files.download(mp4)
files.download(mp4.replace(".mp4", ".srt"))

### Dicas
- Se um slide sair estranho, edite `/content/saida/<nome>/roteiro_usado.md`, salve como `.md` em `/content/entrada` e rode a célula 5 de novo: só os trechos alterados são regravados (cache).
- `exaggeration` maior = mais expressivo; `cfg_weight` menor = mais expressivo, maior = mais fiel ao ritmo da referência.
- Para outro motor: troque `chatterbox-ptbr` por `qwen3` (instale com `%pip install qwen-tts`).
- Clone somente a sua própria voz ou vozes com autorização expressa.